# Script 3 — Sequence spaces

This notebook describes how phee-call sequence repertoires were represented and visualised. Each point is one focal-individual × conspecific × stage × session repertoire containing at least five multi-call sequences.

The default analysis loads the saved distance matrices and fixed two-dimensional coordinates used for the project figures. The optional calculations at the end are deliberately switched off because local alignment is costly.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Run the notebook either from the repository root or from code/.
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "code":
    PROJECT_DIR = PROJECT_DIR.parent

SEQUENCE_DIR = PROJECT_DIR / "data" / "sequence"
DISTANCE_DIR = SEQUENCE_DIR / "distances"
FIGURE_INPUT_DIR = SEQUENCE_DIR / "figure_inputs"
RESULTS_DIR = PROJECT_DIR / "results" / "sequence_spaces"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Expensive calculations are opt-in. The normal reproduction route leaves these False.
RECOMPUTE_SEQUENCE_DISTANCES = False
RECOMPUTE_EMBEDDINGS = False
MAKE_FIGURES = True

METRICS = {
    "local_alignment": "Local alignment",
    "bigram": "Bigram frequency",
    "phee_repeat": "Phee-repeat profile",
    "transition_probability": "Transition probability",
}

MATRIX_FILES = {
    "local_alignment": DISTANCE_DIR / "local_alignment_107.npy",
    "bigram": DISTANCE_DIR / "bigram_107.npy",
    "phee_repeat": DISTANCE_DIR / "phee_repeat_107.npy",
    "transition_probability": DISTANCE_DIR / "transition_probability_107.npy",
}


## Load the sequence repertoires

The processed sequence table retains the original call-level information. `multi_call_sequences_1619.csv` contains the cleaned multi-call sequences, while `session_order_107.csv` gives the exact repertoire order used by every saved matrix and by the Bayesian analysis.


In [ ]:
processed_sequences = pd.read_csv(SEQUENCE_DIR / "sequences_processed.csv")
multi_call_sequences = pd.read_csv(SEQUENCE_DIR / "multi_call_sequences_1619.csv")
sessions = pd.read_csv(SEQUENCE_DIR / "session_order_107.csv").sort_values("group_id").reset_index(drop=True)

if len(sessions) != 107:
    raise ValueError(f"Expected 107 eligible repertoires, found {len(sessions)}")
if not np.array_equal(sessions["group_id"].to_numpy(), np.arange(107)):
    raise ValueError("group_id must run from 0 to 106 in matrix order")
if sessions["group_id"].duplicated().any():
    raise ValueError("Each repertoire must have a unique group_id")

print(f"Processed sequence rows: {len(processed_sequences):,}")
print(f"Cleaned multi-call sequences: {len(multi_call_sequences):,}")
print(f"Eligible session repertoires: {len(sessions):,}")
print(f"Sequences retained in eligible repertoires: {int(sessions['n_seq'].sum()):,}")
display(
    sessions[[
        "group_id", "focal ID", "conspecific_ID", "stage", "paired_status",
        "session_number", "n_seq", "n_unique_sequences",
    ]].head(10)
)


## Load and check the four distance matrices

The four representations measure different aspects of repertoire structure:

- **Local alignment:** average Smith–Waterman similarity across every pair of sequences, converted to distance.
- **Bigram frequency:** Euclidean distance between relative frequencies of adjacent call-type pairs.
- **Phee-repeat profile:** Euclidean distance between the frequencies of Phee runs of length 2–5.
- **Transition probability:** Euclidean distance between row-normalised call-transition matrices.

The checks below are intentionally short and visible: every matrix must contain the same 107 repertoires, be symmetric, have finite non-negative values, and have a zero diagonal.


In [ ]:
distance_matrices = {}
validation_rows = []

for metric, path in MATRIX_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Missing saved matrix: {path}. Download the precomputed data before running this notebook."
        )
    matrix = np.load(path)
    if matrix.shape != (len(sessions), len(sessions)):
        raise ValueError(f"{metric}: expected a 107 x 107 matrix, found {matrix.shape}")
    if not np.isfinite(matrix).all():
        raise ValueError(f"{metric}: matrix contains non-finite values")
    if float(matrix.min()) < -1e-7:
        raise ValueError(f"{metric}: matrix contains negative distances")
    if not np.allclose(matrix, matrix.T, atol=1e-6):
        raise ValueError(f"{metric}: matrix is not symmetric")
    if not np.allclose(np.diag(matrix), 0.0, atol=1e-6):
        raise ValueError(f"{metric}: diagonal is not zero")

    distance_matrices[metric] = matrix
    validation_rows.append({
        "metric": METRICS[metric],
        "repertoires": matrix.shape[0],
        "minimum": float(matrix.min()),
        "maximum": float(matrix.max()),
        "mean_upper_triangle": float(matrix[np.triu_indices_from(matrix, k=1)].mean()),
    })

display(pd.DataFrame(validation_rows))


## Agreement among sequence representations

Correlations are calculated over the upper triangle of each matrix, so every repertoire pair appears once. These correlations describe agreement between representations; they are not the inferential tests used for stage or social context.


In [ ]:
upper = np.triu_indices(len(sessions), k=1)
condensed = pd.DataFrame({METRICS[name]: matrix[upper] for name, matrix in distance_matrices.items()})
sequence_distance_correlations = condensed.corr(method="pearson")
display(sequence_distance_correlations.round(3))


## Load the fixed sequence-space coordinates

Two-dimensional embeddings can move slightly across software versions. The repository therefore includes the exact coordinates used for the figures. Their repertoire identifiers and metadata are checked against `session_order_107.csv` before plotting.


In [ ]:
embeddings = {}
configs = {}
examples = {}
required_metadata = [
    "group_id", "focal ID", "conspecific_ID", "stage", "paired_status",
    "session_number", "pair_id", "sex",
]

for metric in METRICS:
    embedding_path = FIGURE_INPUT_DIR / f"{metric}_embedding.csv"
    config_path = FIGURE_INPUT_DIR / f"{metric}_config.json"
    examples_path = FIGURE_INPUT_DIR / f"{metric}_examples.csv"
    if not embedding_path.exists() or not config_path.exists() or not examples_path.exists():
        raise FileNotFoundError(f"Missing fixed figure input for {metric}")

    embedding = pd.read_csv(embedding_path).sort_values("group_id").reset_index(drop=True)
    if not np.array_equal(embedding["group_id"].to_numpy(), sessions["group_id"].to_numpy()):
        raise ValueError(f"{metric}: embedding order does not match the repertoire order")
    if not np.isfinite(embedding[["embedding_1", "embedding_2"]].to_numpy()).all():
        raise ValueError(f"{metric}: embedding contains non-finite coordinates")
    for column in required_metadata:
        left = embedding[column].astype(str).to_numpy()
        right = sessions[column].astype(str).to_numpy()
        if not np.array_equal(left, right):
            raise ValueError(f"{metric}: {column} does not match session_order_107.csv")

    embeddings[metric] = embedding
    configs[metric] = json.loads(config_path.read_text())
    examples[metric] = pd.read_csv(examples_path)
    if not set(examples[metric]["group_id"].astype(int)).issubset(set(sessions["group_id"])):
        raise ValueError(f"{metric}: example table contains unknown group_id values")

print("Fixed coordinates match all 107 ordered repertoires for all four metrics.")


## Recreate the final sequence-space figures

These are the last figures produced by the sequence-space workflow. Panel A links selected points to metric-specific examples: complete exact-sequence repertoires for local alignment, bigram and transition heatmaps, and Phee-run profiles. Panel B separates stage and social context and adds descriptive ellipses containing the central 50% of each individual’s observed spread in the displayed two-dimensional space.

The 50% ellipses describe the displayed points; they are not confidence intervals or hypothesis tests. The fixed coordinates, example selections, subset counts, and layout settings are included in `data/sequence/figure_inputs/`. The focused `sequence_distribution_figure.py` helper contains the detailed card and panel layout so that this notebook remains readable.


In [ ]:
if MAKE_FIGURES:
    import sys

    code_directory = str((PROJECT_DIR / "code").resolve())
    if code_directory not in sys.path:
        sys.path.insert(0, code_directory)
    from sequence_distribution_figure import reproduce_sequence_distribution_figures

    reproduced = reproduce_sequence_distribution_figures(PROJECT_DIR)
    print("Saved the four final 50% distribution figures:")
    for path in reproduced.values():
        print(" -", path.relative_to(PROJECT_DIR))
else:
    print("Figure creation is switched off.")


## Optional: recalculate the sequence distances

The saved matrices above are the normal route. The code below retains the core calculations in a single place for readers who want to rebuild them. It uses the exact ordered repertoires in `session_order_107.csv` and only runs when `RECOMPUTE_SEQUENCE_DISTANCES` is changed to `True`. Local alignment evaluates every sequence pair and is substantially slower than the other three representations.


In [ ]:
if RECOMPUTE_SEQUENCE_DISTANCES:
    import math
    from itertools import combinations, product

    if "sequences_json" not in sessions:
        raise KeyError("session_order_107.csv must contain sequences_json for recalculation")
    repertoires = sessions["sequences_json"].map(json.loads).tolist()
    alphabet = sorted({token for repertoire in repertoires for sequence in repertoire for token in sequence})

    def bigram_vector(repertoire):
        vocabulary = list(product(alphabet, repeat=2))
        counts = {pair: 0 for pair in vocabulary}
        for sequence in repertoire:
            for pair in zip(sequence[:-1], sequence[1:]):
                counts[pair] += 1
        denominator = sum(counts.values()) + 1e-3
        return np.array([counts[pair] / denominator for pair in vocabulary])

    def phee_repeat_vector(repertoire):
        counts = {length: 0 for length in range(2, 6)}
        for sequence in repertoire:
            position = 0
            while position < len(sequence):
                end = position + 1
                while end < len(sequence) and sequence[end] == sequence[position]:
                    end += 1
                run_length = end - position
                if sequence[position] == "A" and run_length in counts:
                    counts[run_length] += 1
                position = end
        denominator = sum(counts.values()) + 1e-3
        return np.array([counts[length] / denominator for length in range(2, 6)])

    def transition_vector(repertoire):
        counts = np.zeros((len(alphabet), len(alphabet)), dtype=float)
        lookup = {token: index for index, token in enumerate(alphabet)}
        for sequence in repertoire:
            for first, second in zip(sequence[:-1], sequence[1:]):
                counts[lookup[first], lookup[second]] += 1
        row_totals = counts.sum(axis=1, keepdims=True)
        return np.divide(counts, row_totals, out=np.zeros_like(counts), where=row_totals > 0).ravel()

    def euclidean_matrix(vectors):
        vectors = np.vstack(vectors)
        differences = vectors[:, None, :] - vectors[None, :, :]
        return np.sqrt(np.sum(differences * differences, axis=2)).astype(np.float32)

    def smith_waterman(sequence_a, sequence_b, match=2, mismatch=-1, gap=-1):
        previous = np.zeros(len(sequence_b) + 1, dtype=int)
        best = 0
        for token_a in sequence_a:
            current = np.zeros(len(sequence_b) + 1, dtype=int)
            for column, token_b in enumerate(sequence_b, start=1):
                substitution = match if token_a == token_b else mismatch
                current[column] = max(
                    0,
                    previous[column - 1] + substitution,
                    previous[column] + gap,
                    current[column - 1] + gap,
                )
                best = max(best, int(current[column]))
            previous = current
        return best

    def repertoire_similarity(first, second):
        values = [
            smith_waterman(sequence_a, sequence_b) / math.log(len(sequence_a) + len(sequence_b))
            for sequence_a in first for sequence_b in second
        ]
        return float(np.mean(values))

    recalculated = {
        "bigram": euclidean_matrix([bigram_vector(values) for values in repertoires]),
        "phee_repeat": euclidean_matrix([phee_repeat_vector(values) for values in repertoires]),
        "transition_probability": euclidean_matrix([transition_vector(values) for values in repertoires]),
    }

    similarities = np.zeros((len(repertoires), len(repertoires)), dtype=np.float32)
    for first, second in combinations(range(len(repertoires)), 2):
        value = repertoire_similarity(repertoires[first], repertoires[second])
        similarities[first, second] = similarities[second, first] = value
    maximum_similarity = float(similarities.max())
    np.fill_diagonal(similarities, maximum_similarity)
    recalculated["local_alignment"] = maximum_similarity - similarities
    np.fill_diagonal(recalculated["local_alignment"], 0.0)

    for metric, matrix in recalculated.items():
        output = DISTANCE_DIR / f"{metric}_107_recalculated.npy"
        np.save(output, matrix.astype(np.float32))
        print("Saved", output.relative_to(PROJECT_DIR))
else:
    print("Using the included sequence-distance matrices; no distances were recalculated.")


## Optional: refit the two-dimensional embeddings

The publication figures use the included fixed coordinates. This optional block creates new UMAP coordinates from the saved distances without overwriting the published inputs. Small differences from the included coordinates are expected across UMAP versions.


In [ ]:
if RECOMPUTE_EMBEDDINGS:
    import umap

    for metric, matrix in distance_matrices.items():
        coordinates = umap.UMAP(
            n_neighbors=30,
            min_dist=0.10,
            metric="precomputed",
            random_state=42,
        ).fit_transform(matrix)
        output = sessions.copy()
        output["embedding_1"] = coordinates[:, 0]
        output["embedding_2"] = coordinates[:, 1]
        output["embedding_method"] = "UMAP"
        path = FIGURE_INPUT_DIR / f"{metric}_embedding_recalculated.csv"
        output.to_csv(path, index=False)
        print("Saved", path.relative_to(PROJECT_DIR))
else:
    print("Using the included fixed embedding coordinates.")
